# ETL: Load Training/Validation Splits

This notebook loads training and validation split assignments into the database.

**Prerequisites:**
- Images must already be loaded in the database
- Analysis run must already exist
- CSV files with filepaths must be prepared

In [ ]:
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import sys
import os
import re
# Add parent directory to path to import db_loader
sys.path.append('..')
from source.db_loader import MLDataLoader

In [ ]:
def extract_image_id_from_filepath(df, filepath_column='file_paths', id_column='image_id'):
    """
    Extract image_id from file paths by getting the trailing integer from filename.
    
    Purpose:
    --------
    Creates an image_id column by extracting the integer at the end of each filename.
    For example: "BernerOberland001.tif" -> 1, "map_0042.jpg" -> 42
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing file paths
    filepath_column : str, optional
        Name of column containing file paths (default: 'file_paths')
    id_column : str, optional
        Name of new column to create (default: 'image_id')
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with new image_id column
    
    How it works:
    -------------
    Uses regex to find digits immediately before the file extension
    Pattern: (\d+)(?=\.\w+$) means "capture digits that come before .extension"
    
    Raises:
    -------
    ValueError: If any filepath doesn't have a trailing integer
    """
    
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Extract integer from each filepath
    def get_id_from_path(filepath):
        if pd.isna(filepath):
            return None
        
        # Get just the filename (remove directory path)
        filename = os.path.basename(filepath)
        
        # Extract trailing integer before file extension
        # Example: "BernerOberland001.tif" -> captures "001"
        match = re.search(r'(\d+)(?=\.\w+$)', filename)
        
        if match:
            return int(match.group(1))
        else:
            raise ValueError(f"Could not extract image_id from filename: {filename}")
    
    # Apply to all rows
    df[id_column] = df[filepath_column].apply(get_id_from_path)
    
    # Report results
    n_extracted = df[id_column].notna().sum()
    print(f"✅ Extracted {n_extracted} image_ids from file paths")
    
    if df[id_column].isna().any():
        n_missing = df[id_column].isna().sum()
        print(f"⚠️  {n_missing} file paths did not have extractable image_ids")
    
    return df




In [ ]:
def fix_filepaths_if_needed(df, image_dir, file_source, filepath_column='file_paths'):
    """
    Fix file paths in dataframe if they point to .jpg files instead of .tif originals.
    
    Purpose:
    --------
    During analysis, images may be converted to .jpg format and stored separately.
    The database only stores original .tif files. This function checks if the 
    file_paths column contains .jpg paths and remaps them to .tif originals if needed.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing 'image_id' and filepath column
    image_dir : Path or str
        Directory containing the original .tif image files
    file_source : str
        Source identifier (e.g., 'giub') to add to dataframe
    filepath_column : str, optional
        Name of the column containing file paths (default: 'file_paths')
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with corrected file paths pointing to .tif files
    
    How it works:
    -------------
    1. Check first file path - if it's .tif, no changes needed (return as-is)
    2. If it's .jpg, build mapping from image_id to .tif file paths
    3. Remap all file paths using image_id
    4. Warn about any unmatched IDs
    """
    
    # Make a copy to avoid modifying original
    df = df.copy()
    
    # Check if remapping is needed (look at first non-null filepath)
    first_path = df[filepath_column].dropna().iloc[0] if len(df[filepath_column].dropna()) > 0 else None
    
    if first_path is None:
        print("⚠️  No file paths found in dataframe")
        return df
    
    if first_path.lower().endswith('.tif'):
        print("✅ File paths already point to .tif files - no remapping needed")
        df['file_source'] = file_source
        return df
    
    # File paths are .jpg - need to remap to .tif originals
    print("📝 File paths are .jpg - remapping to .tif originals...")
    
    # Get all .tif files in image directory
    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith('.tif')]
    print(f"   Found {len(image_files)} .tif files in {image_dir}")
    
    # Build mapping: image_id -> full .tif file path
    # Extract trailing number from filename (e.g., "BernerOberland001.tif" -> 1)
    id_to_filepath = {}
    for filename in image_files:
        match = re.search(r'(\d+)(?=\.\w+$)', filename)
        if match:
            image_id = int(match.group(1))
            id_to_filepath[image_id] = str(image_dir / filename)
        else:
            print(f"   Warning: could not extract image_id from filename: {filename}")
    
    print(f"   Mapped {len(id_to_filepath)} filenames to image_ids")
    
    # Remap file paths using image_id
    df[filepath_column] = df['image_id'].map(id_to_filepath)
    df['file_source'] = file_source
    
    # Check for unmatched IDs
    n_missing = df[filepath_column].isna().sum()
    if n_missing > 0:
        print(f"   ⚠️  {n_missing} image_ids could not be matched to .tif files")
        print(df[df[filepath_column].isna()][['image_id', filepath_column]])
    else:
        print(f"   ✅ All {len(df)} image_ids successfully matched to .tif files")
    
    return df



In [ ]:
#database_name = 'image_analysis_dev'
database_name = 'image_analysis_test'

#db_env = '.env'
db_env = '.env.test'

# Load test environment (override=True ensures it replaces any previously loaded values)
load_dotenv(db_env, override=True)

# Explicit check — you see exactly what you're connecting to
db_name = os.getenv('DB_NAME')
print(f"Connecting to: {db_name}")
assert db_name == database_name, f"Wrong database loaded: {db_name}"


print("✅ Libraries loaded")

In [ ]:
# Connect to database
loader = MLDataLoader(db_name='image_analysis_test', source=None)

### Set file paths:

In [ ]:
# Configure paths and analysis run ID
#data_path = Path('/Users/stephanehess/Documents/CAS_AML/dias_digit_project')
project_path = Path.cwd()
#data_path = (project_path/ '..' /'data_folders'/'combined_data').resolve()
data_path = (project_path/ '..' /'data_folders'/'data').resolve()
image_dir = (project_path/ '..' /'data_folders'/'data_1').resolve()


In [ ]:
sorted(os.listdir(data_path), key=lambda f: os.path.getmtime(os.path.join(data_path, f)), reverse=True)


### IMPORTANT: Set the analysis_run_id for your clustering run

In [ ]:
# IMPORTANT: Set the analysis_run_id for your clustering run
analysis_run_id = 11  # TODO: Change this to your actual analysis_run_id

### Set file source: 

In [ ]:
file_source = 'giub'

### Set file names: 

In [ ]:
#train_csv = data_path / '20260314_211803/train_data_file_paths.csv'
#train_csv = data_path / '20260324_233333/train_data_file_paths_20260324_233333.csv'
#train_csv = data_path / '20260325_110746/train_data_file_paths_20260325_110746.csv'
train_csv = data_path / '20260325_201946/train_data_file_paths_20260325_201946.csv'
#train_csv = data_path / 'train_data_file_paths_20260325_204829.csv'
#val_csv = data_path / '20260314_211803/val_data_file_paths.csv'
#val_csv = data_path / '20260324_233333/val_data_file_paths_20260324_233333.csv'
#val_csv = data_path / '20260325_110746/val_data_file_paths_20260325_110746.csv'
val_csv = data_path / '20260325_201946/val_data_file_paths_20260325_201946.csv'
#val_csv = data_path / 'val_data_file_paths_20260325_204829.csv'

print(f"Loading splits for analysis_run_id: {analysis_run_id}")
print(f"Train CSV: {train_csv}")
print(f"Val CSV: {val_csv}")

## Load Training Data

In [ ]:
# Load training filepaths from CSV
train_df = pd.read_csv(train_csv)
print(f"Loaded {len(train_df)} training filepaths")
print("\nFirst few rows:")
display(train_df.head())

### Add image_ids: 

In [ ]:
train_df.filepaths[0]

In [ ]:
train_df = extract_image_id_from_filepath(
    df=train_df,
    filepath_column='filepaths',  # or 'file_paths' depending on your CSV
    id_column='image_id'
)

In [ ]:
print(train_df.head())
print(train_df.image_id[0])
print(train_df.filepaths[0])

### Replace file paths with file paths to original images if necessary:

In [ ]:
train_df = fix_filepaths_if_needed(
    df=train_df,
    image_dir=image_dir,
    file_source=file_source,
    filepath_column='filepaths'  # Note: might be 'filepaths' not 'file_paths'
)


train_df.filepaths[0]

### Load data into database:

In [ ]:

# Extract filepaths as list
train_filepaths = train_df['filepaths'].tolist()

# Load into database
train_result = loader.load_training_validation_splits(
    filepaths=train_filepaths,
    split_type='train',
    analysis_run_id=analysis_run_id
)

print(f"\nResult: {train_result}")

## Load Validation Data

In [ ]:
# Load validation filepaths from CSV
val_df = pd.read_csv(val_csv)
print(f"Loaded {len(val_df)} validation filepaths")
print("\nFirst few rows:")
display(val_df.head())

### Add image_ids:

In [ ]:
val_df.filepaths[0]

In [ ]:
val_df = extract_image_id_from_filepath(
    df=val_df,
    filepath_column='filepaths',  # or 'file_paths' depending on your CSV
    id_column='image_id'
)

In [ ]:
print(val_df.head())
print(val_df.image_id[0])
print(val_df.filepaths[0])

### Replace file paths with file paths to original images if necessary:

In [ ]:
val_df = fix_filepaths_if_needed(
    df=val_df,
    image_dir=image_dir,
    file_source=file_source,
    filepath_column='filepaths'  # Note: might be 'filepaths' not 'file_paths'
)


In [ ]:
val_df.filepaths[0]

### Load data into database:

In [ ]:


# Extract filepaths as list
val_filepaths = val_df['filepaths'].tolist()

# Load into database
val_result = loader.load_training_validation_splits(
    filepaths=val_filepaths,
    split_type='validation',
    analysis_run_id=analysis_run_id
)

print(f"\nResult: {val_result}")

## Verify Data Loading

In [ ]:
# Query to check what was loaded
import psycopg2

verification = pd.read_sql(f"""
    SELECT 
        split_type,
        COUNT(*) as count
    FROM training_validation_splits
    WHERE analysis_run_id = {analysis_run_id}
    GROUP BY split_type
    ORDER BY split_type
""", loader.conn)

print("\nSplit counts in database:")
display(verification)

# Calculate split ratio
if len(verification) == 2:
    train_count = verification[verification['split_type'] == 'train']['count'].values[0]
    val_count = verification[verification['split_type'] == 'validation']['count'].values[0]
    total = train_count + val_count
    train_pct = (train_count / total) * 100
    val_pct = (val_count / total) * 100
    print(f"\nSplit ratio: {train_pct:.1f}% train / {val_pct:.1f}% validation")

In [ ]:
# Close database connection
loader.close()

In [ ]:
7000/60

In [ ]:
8000/60